# 유저 리텐션 영향 인자 분석 및 가설 검정
> **목적**: 초보 유저의 이탈에 가장 큰 영향을 미치는 요인(봇 비중 등)을 통계적으로 규명합니다.

### 주요 분석 스토리라인:
1. **가설 설정**: "봇 노출도가 높을수록 실제 유저와의 교감이 적어 이탈률이 높아질 것이다."
2. **상관 분석**: `is_retained`와 각 환경 변수 간의 Spearman 상관계수 산출
3. **통계 검정 (핵심)**: 
   - Shapiro-Wilk 검정으로 데이터의 비정규성 확인
   - T-test 결과의 불충분함 인지 후 **Mann-Whitney U Test** 최종 채택
4. **인사이트 도출**: 리텐션 그룹 간의 통계적 차이 증명 및 비즈니스 개선 제언

In [ ]:
# STEP 0. 환경 설정

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'Malgun Gothic'

In [ ]:
# STEP 1. 데이터 로드

df = pd.read_csv('data/processed/user_final.csv')

df.head()

In [ ]:
# STEP 2. 전체 유저 현황 파악

# 전체 유저 수
df['userID'].nunique()

# 평균 플레이 횟수
df['total_games'].mean()

# 리텐션 비율
df['is_retained'].mean()

In [ ]:
# STEP 3. 초보 vs 전체 유저 비교

novice = df[df['is_novice'] == 1]
normal = df[df['is_novice'] == 0]

# 평균 비교
compare = pd.DataFrame({
    '전체': df.mean(numeric_only=True),
    '초보': novice.mean(numeric_only=True)
})

compare

In [ ]:
# STEP 4. 핵심 지표 분포 분석

# 플레이 횟수 분포
sns.histplot(df['total_games'], bins=50)
plt.show()

# 킬 수 분포
sns.boxplot(x=df['kill'])
plt.show()

In [ ]:
# STEP 5. 리텐션 분석

# 리텐션 vs 비리텐션 비교
sns.boxplot(x='is_retained', y='total_games', data=df)
plt.show()

# 봇 비율 영향
sns.boxplot(x='is_retained', y='bot_exposure', data=df)
plt.show()

# STEP 6. 가설 설정

"""
가설 1: 초보 유저는 봇 비율이 높을수록 이탈한다
가설 2: 플레이 경험(게임 수)이 많을수록 리텐션이 높다
가설 3: 실제 유저와의 전투 경험이 부족하면 이탈한다
"""

# STEP 7. 통계 검정

from scipy.stats import ttest_ind, mannwhitneyu, f_oneway, kruskal

retained = df[df['is_retained'] == 1]
churned = df[df['is_retained'] == 0]

# 1️. 정규성 무시하고 t-test 시도
t_stat, p_val = ttest_ind(retained['bot_exposure'], churned['bot_exposure'])

# 2️. 비모수 ANOVA (Kruskal)
k_stat, k_p = kruskal(retained['bot_exposure'], churned['bot_exposure'])

# 최종 선택: Mann-Whitney U Test
u_stat, u_p = mannwhitneyu(retained['bot_exposure'], churned['bot_exposure'])

print(u_p)

# STEP 8. 인사이트 도출

"""
- 봇 비율이 높은 유저일수록 리텐션이 낮음
- 실제 유저와의 상호작용 부족 → 흥미 감소
- 초반 경험이 중요
"""

# STEP 9. 비즈니스 액션

"""
1. 초보 구간에서 실제 유저 매칭 증가
2. 봇 난이도 조정
3. 초반 성취 경험 강화 (킬 유도)
"""